# Taller 3 — APIs Públicas, MongoDB y EDA
## Personajes de *Bleach* ⚔️

**Autora:** Daniela Alejandra Salazar
**Curso:** Bases de Datos para Ciencia de Datos — Docente: Miguel Ramos García

**API usada:** [Bleach API](https://bleach-api-8v2r.onrender.com) (pública, gratuita y sin llave), construida a partir de la Bleach Fandom Wiki.
Agrupa a los personajes en cuatro categorías: **shinigami, humans, quincy y arrancar**.

**Flujo del proyecto:**
1. `ingesta.py` descarga los personajes de la API y guarda el JSON crudo en MongoDB (`taller3_db.raw_data`).
2. Este notebook lee esa colección, selecciona variables relevantes, las limpia y hace el análisis exploratorio (EDA).

### 1. Librerías
Importamos `pymongo` para conectarnos a MongoDB, `pandas` para el manejo de los datos y `matplotlib` para las gráficas.

In [ ]:
import os
import re

import pandas as pd
import matplotlib.pyplot as plt
from dotenv import load_dotenv
from pymongo import MongoClient

pd.set_option("display.max_columns", None)
plt.rcParams["figure.dpi"] = 100

### 2. Conexión a MongoDB y lectura de los datos crudos
Nos conectamos a la base de datos `taller3_db` y leemos todos los documentos de la colección `raw_data`, que fueron cargados por `ingesta.py` sin modificar.

In [ ]:
load_dotenv()
MONGO_URI = os.getenv("MONGO_URI", "mongodb://localhost:27017")

client = MongoClient(MONGO_URI)
coleccion = client["taller3_db"]["raw_data"]

print("Documentos en taller3_db.raw_data:", coleccion.count_documents({}))
documentos = list(coleccion.find())

Así se ve un documento crudo (se muestran solo sus primeras claves). Cada personaje trae su nombre, descripción, imágenes, citas y un diccionario anidado `stats` con raza, género, estatura, peso, zanpakutō, primera aparición, actores de voz, etc.

In [ ]:
ejemplo = coleccion.find_one({"_id": "byakuya_kuchiki"}) or documentos[0]
print("Claves del documento:", list(ejemplo.keys()))
print("Claves de 'stats':   ", list(ejemplo["stats"].keys()))
ejemplo["stats"]

### 3. Selección de variables
El JSON crudo es muy anidado. Para el análisis seleccionamos **10 variables** relevantes:

| Columna | Origen en el JSON | Descripción |
|---|---|---|
| `nombre` | `name.english` | Nombre del personaje |
| `categoria` | `_categoria_api` | Categoría de la API (Shinigami, Humano, Quincy, Arrancar) |
| `raza` | `stats.race` | Raza principal |
| `genero` | `stats.gender` | Género |
| `altura_cm` | `stats.height` | Estatura en centímetros |
| `peso_kg` | `stats.weight` | Peso en kilogramos |
| `capitulo_debut` | `stats.First Appearance.manga` | Capítulo del manga en que aparece por primera vez |
| `num_citas` | `quotes` | Cantidad de citas célebres registradas |
| `num_imagenes` | `media` | Cantidad de imágenes en la wiki (aprox. de su relevancia) |
| `tiene_bankai` | `stats.Zanpakutō.bankai` | Si el personaje tiene un bankai con nombre revelado |

Los campos de texto como `"180 cm (5'11\")"` se convierten a número con expresiones regulares.

In [ ]:
def primer_numero(texto, unidad):
    """Extrae el primer número que aparece antes de la unidad: '180 cm (5'11")' -> 180.0"""
    if not isinstance(texto, str):
        return None
    m = re.search(r"(\d+(?:\.\d+)?)\s*" + unidad, texto)
    return float(m.group(1)) if m else None


def capitulo(texto):
    """'Volume 6, Chapter 51' -> 51. Los capítulos negativos (flashbacks) se ignoran."""
    if isinstance(texto, list):
        texto = texto[0]
    if not isinstance(texto, str):
        return None
    m = re.search(r"Chapter\s+(\d+)", texto)
    return int(m.group(1)) if m else None


def bankai_revelado(zanpakuto):
    """True si el bankai tiene nombre; False si es desconocido / no alcanzado / no aplica."""
    bankai = (zanpakuto or {}).get("bankai")
    if not isinstance(bankai, str):
        return False
    no_revelado = ("unknown", "none", "not", "unnamed", "details", "n/a")
    return not bankai.strip().lower().startswith(no_revelado)


def a_texto(valor):
    return "; ".join(valor) if isinstance(valor, list) else valor


filas = []
for doc in documentos:
    stats = doc.get("stats", {})
    filas.append({
        "id": doc["_id"],
        "nombre": doc.get("name", {}).get("english"),
        "categoria": doc.get("_categoria_api"),
        "raza": a_texto(stats.get("race")),
        "genero": a_texto(stats.get("gender")),
        "altura_cm": primer_numero(stats.get("height"), "cm"),
        "peso_kg": primer_numero(stats.get("weight"), "kg"),
        "capitulo_debut": capitulo(stats.get("First Appearance", {}).get("manga")),
        "num_citas": len(doc.get("quotes") or []),
        "num_imagenes": len(doc.get("media") or []),
        "tiene_bankai": bankai_revelado(stats.get("Zanpakutō")),
    })

df = pd.DataFrame(filas)
print("Filas x columnas:", df.shape)

### 4. Limpieza
- La API incluye algunas **páginas grupales** que no son un personaje individual (por ejemplo *"Miscellaneous Gotei 13 Members"*, *"Ōmaeda Clan"* o *"Karakura High School Students"*). Las eliminamos.
- Traducimos las categorías y los géneros al español. Los géneros distintos de Masculino/Femenino (ej. *Genderless*, *Unknown*) quedan como **Otro / Desconocido**.
- La raza puede traer varias (ej. `"Human; Fullbringer"`); nos quedamos con la principal.
- `capitulo_debut` se guarda como entero que admite nulos (`Int64`), porque algunos personajes solo aparecen en el anime o en videojuegos.

In [ ]:
paginas_grupales = df["id"].str.contains(r"miscellaneous|_clan|students|cloning_|hammerhead_|lumina_and", regex=True)
print("Páginas grupales eliminadas:", df.loc[paginas_grupales, "nombre"].tolist())
df = df[~paginas_grupales].copy()

df["categoria"] = df["categoria"].map({
    "shinigami": "Shinigami", "humans": "Humano", "quincy": "Quincy", "arrancar": "Arrancar"
})
df["genero"] = df["genero"].map({"Male": "Masculino", "Female": "Femenino"}).fillna("Otro / Desconocido")
df["raza"] = df["raza"].str.split(r"[;(]").str[0].str.strip()
df["capitulo_debut"] = df["capitulo_debut"].astype("Int64")

df = df.drop(columns="id").reset_index(drop=True)
print("DataFrame limpio:", df.shape)

### 5. Inspección básica
#### 5.1 Primeras filas — `head()`

In [ ]:
df.head(10)

#### 5.2 Tipos de datos — `info()`

In [ ]:
df.info()

#### 5.3 Valores nulos
Las variables físicas (`altura_cm`, `peso_kg`) y `capitulo_debut` tienen nulos porque la wiki no registra ese dato para todos los personajes (sobre todo secundarios o exclusivos del anime). **No los rellenamos**: los cálculos de promedios simplemente los excluyen.

In [ ]:
nulos = pd.DataFrame({
    "nulos": df.isnull().sum(),
    "% nulos": (df.isnull().mean() * 100).round(1),
})
nulos

#### 5.4 Estadísticas descriptivas de las variables numéricas

In [ ]:
df.describe().round(1)

### 6. Insights numéricos (5 datos relevantes)
#### Insight 1 — ¿Cuántos personajes hay por categoría?

In [ ]:
por_categoria = df["categoria"].value_counts()
print(por_categoria.to_string(), "\n")
print(f"Total de personajes analizados: {len(df)}")
print(f"Los Shinigami son el {por_categoria['Shinigami'] / len(df):.1%} de todos los personajes.")

#### Insight 2 — Distribución por género

In [ ]:
genero = df["genero"].value_counts()
genero_pct = (df["genero"].value_counts(normalize=True) * 100).round(1)
print(pd.DataFrame({"personajes": genero, "%": genero_pct}).to_string())
print(f"\nHay {genero['Masculino']} personajes masculinos y {genero['Femenino']} femeninos "
      f"({genero['Masculino'] / genero['Femenino']:.1f} hombres por cada mujer).")

#### Insight 3 — Estatura: promedio, el más alto y el más bajo

In [ ]:
print(f"Estatura promedio: {df['altura_cm'].mean():.1f} cm "
      f"(mediana {df['altura_cm'].median():.0f} cm, con dato en {df['altura_cm'].notna().sum()} personajes)\n")

mas_alto = df.loc[df["altura_cm"].idxmax()]
mas_bajo = df.loc[df["altura_cm"].idxmin()]
print(f"Más alto: {mas_alto['nombre']} ({mas_alto['categoria']}) con {mas_alto['altura_cm']:.0f} cm")
print(f"Más bajo: {mas_bajo['nombre']} ({mas_bajo['categoria']}) con {mas_bajo['altura_cm']:.0f} cm\n")

print("Estatura promedio por categoría (cm):")
print(df.groupby("categoria")["altura_cm"].mean().round(1).sort_values(ascending=False).to_string())

#### Insight 4 — Los personajes más relevantes en la wiki (más imágenes y más citas)

In [ ]:
top_imagenes = df.nlargest(5, "num_imagenes")[["nombre", "categoria", "num_imagenes"]]
print("Top 5 por número de imágenes:")
print(top_imagenes.to_string(index=False), "\n")

top_citas = df.nlargest(1, "num_citas").iloc[0]
print(f"Personaje con más citas célebres: {top_citas['nombre']} ({top_citas['num_citas']} citas)")
print(f"Promedio de citas por personaje: {df['num_citas'].mean():.1f}")

#### Insight 5 — ¿Cuántos Shinigami tienen un bankai revelado?

In [ ]:
shinigami = df[df["categoria"] == "Shinigami"]
con_bankai = shinigami["tiene_bankai"].sum()
print(f"De {len(shinigami)} Shinigami, solo {con_bankai} tienen un bankai con nombre revelado "
      f"({con_bankai / len(shinigami):.1%}).")
print("\nAlgunos de ellos:", ", ".join(shinigami.loc[shinigami['tiene_bankai'], 'nombre'].head(8)))

#### Insight extra — ¿En qué parte del manga debuta cada categoría?

In [ ]:
debut = df.groupby("categoria")["capitulo_debut"].agg(["median", "min", "max"]).sort_values("median")
debut.columns = ["capítulo mediano", "primero", "último"]
debut

### 7. Visualizaciones
#### Gráfico 1 (torta) — Proporción de personajes por género

In [ ]:
fig, ax = plt.subplots(figsize=(7, 7))
colores = ["#4C72B0", "#DD8452", "#A5A5A5"]
genero.plot.pie(
    ax=ax, autopct="%1.1f%%", startangle=90, colors=colores[: len(genero)],
    wedgeprops={"edgecolor": "white"}, textprops={"fontsize": 12},
)
ax.set_ylabel("")
ax.set_title("Bleach: proporción de personajes por género", fontsize=14)
plt.tight_layout()
plt.show()

#### Gráfico 2 (barras) — Estatura promedio por categoría

In [ ]:
altura_cat = df.groupby("categoria")["altura_cm"].mean().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(8, 5))
barras = ax.bar(altura_cat.index, altura_cat.values, color=["#2A9D8F", "#E9C46A", "#F4A261", "#E76F51"])
ax.bar_label(barras, fmt="%.0f cm", padding=3)
ax.set_title("Estatura promedio por categoría de personaje", fontsize=14)
ax.set_xlabel("Categoría")
ax.set_ylabel("Estatura promedio (cm)")
ax.set_ylim(0, altura_cat.max() * 1.15)
plt.tight_layout()
plt.show()

#### Gráfico 3 (histograma) — Capítulo del manga en que debutan los personajes

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
for cat in ["Humano", "Shinigami", "Arrancar", "Quincy"]:
    datos = df.loc[df["categoria"] == cat, "capitulo_debut"].dropna()
    ax.hist(datos, bins=range(0, 700, 25), alpha=0.6, label=cat)
ax.set_title("¿En qué capítulo del manga aparece cada personaje por primera vez?", fontsize=14)
ax.set_xlabel("Capítulo de debut en el manga")
ax.set_ylabel("Número de personajes")
ax.legend(title="Categoría")
plt.tight_layout()
plt.show()

### 8. Conclusiones
1. **Los Shinigami dominan el elenco:** son casi la mitad de los personajes de la API, seguidos por los Arrancar, los Humanos y los Quincy.
2. **El reparto es mayoritariamente masculino:** hay unos dos personajes y medio masculinos por cada personaje femenino.
3. **Los Arrancar son en promedio los más altos** y los Humanos los más bajos (muchos humanos son estudiantes o niños). Los dos extremos de estatura son Arrancar: Choe Neng Poww (el más alto) y Kukkapūro (el más bajo, que es un perro).
4. **El protagonismo se nota en los datos:** Ichigo Kurosaki es, por mucho, el personaje con más imágenes en la wiki, casi el doble que el segundo.
5. **El bankai es raro:** solo alrededor de 1 de cada 5 Shinigami tiene un bankai con nombre revelado, y muchos de ellos son capitanes o excapitanes.
6. **Los datos reflejan la estructura de la historia:** los Humanos debutan en los primeros capítulos, los Arrancar a mitad del manga y los Quincy en el arco final (*Thousand-Year Blood War*, alrededor del capítulo 480 en adelante).

In [ ]:
client.close()